In [1]:
import sys
import os
import tqdm
import gc
import torch
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import ListedColormap, BoundaryNorm

module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.append(module_path)
    
from utils import ini_argparse, split_dataset
from dataset import *
from model import MinkUNetConvNeXtV2

import matplotlib
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import font_manager
import matplotlib.colors as mcolors
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt

# reset the plot configurations to default
plt.rcdefaults()

from pathlib import Path
font_path = str(Path(matplotlib.get_data_path(), "fonts/ttf/cmr10.ttf"))
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()
plt.rcParams["axes.formatter.use_mathtext"] = True
params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

/scratch/fcufino/conda_envs/convnextv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/fcufino/conda_envs/convnextv2/lib/python3.10/site-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(


In [2]:
# manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]=""

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
#device = torch.device('cpu')

parser = ini_argparse()
args = parser.parse_args([])
args.train = False
args.dataset_path = "/scratch/salonso/sparse-nns/faser/events_v3.5"
args.sets_path = "/scratch/salonso/sparse-nns/faser/events_v3.5/sets.pkl"
args.batch_size = 16
args.num_workers = 16
args.checkpoint_path = "/scratch3/fcufino/checkpoints_final"

print("\n- Arguments:")
for arg, value in vars(args).items():
    print(f"  {arg}: {value}")
nb_gpus = len(args.gpus)
gpus = [int(gpu) for gpu in args.gpus]

cpu

- Arguments:
  train: False
  stage1: True
  dataset_path: /scratch/salonso/sparse-nns/faser/events_v3.5
  sets_path: /scratch/salonso/sparse-nns/faser/events_v3.5/sets.pkl
  load_seg: False
  eps: 1e-12
  chunk_size: 512
  batch_size: 16
  epochs: 50
  num_workers: 16
  lr: 0.0001
  accum_grad_batches: 1
  warmup_steps: 0
  cosine_annealing_steps: 0
  weight_decay: 0.05
  beta1: 0.9
  beta2: 0.999
  losses: ['focal', 'dice']
  save_dir: /scratch/salonso/sparse-nns/faser/deep_learning/faserDL
  name: v1
  log_every_n_steps: 50
  save_top_k: 1
  checkpoint_path: /scratch3/fcufino/checkpoints_final
  checkpoint_name: v1
  load_checkpoint: None
  gpus: [0]


In [3]:
dataset = SparseFASERCALDataset(args)
print("- Dataset size: {} events".format(len(dataset)))
print(args.sets_path)
train_loader, valid_loader, test_loader = split_dataset(dataset, args, splits=[0.6, 0.1, 0.3], test=True)

- Dataset size: 144939 events
/scratch/salonso/sparse-nns/faser/events_v3.5/sets.pkl
Loaded saved splits!


In [4]:
model = MinkUNetConvNeXtV2(in_channels=1, out_channels=4, D=3, args=args)
checkpoint = torch.load("/scratch3/fcufino/checkpoints_final/seg_v1/last.ckpt", map_location='cpu')

# Remove the "model." prefix from the keys in the state_dict
state_dict = {key.replace("model.", ""): value for key, value in checkpoint['state_dict'].items()}
model.load_state_dict(state_dict, strict=True)

if device.type == 'cpu':
    #model = replace_depthwise_with_channelwise(model)
    model.replace_depthwise_with_channelwise()
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total trainable params model (total): {}".format(total_params))

Total trainable params model (total): 36476160


In [5]:
dataset[0]

{'run_number': 40,
 'event_id': 3332,
 'primary_vertex': array([ 4.96929893e+01, -1.03150785e+00,  1.29868726e+03]),
 'is_cc': False,
 'in_neutrino_pdg': -14,
 'in_neutrino_energy': 217.1,
 'out_lepton_momentum_dir': tensor([0., 0., 0.]),
 'primlepton_labels': tensor([[0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
         [0.],
     

In [6]:
from utils import arrange_sparse_minkowski, arrange_truth
from sklearn.metrics import confusion_matrix as sklearn_confusion_matrix

def _arrange_batch(batch, device):
    batch_input, batch_input_global = arrange_sparse_minkowski(batch, device)
    batch_input_global = batch_input_global.to(device)
    target = arrange_truth(batch)
    return batch_input, batch_input_global, target

def confusion_matrix(predictions, targets, labels=None, round_target=False):
    if predictions.shape[1] == 1:
        pred_classes = np.round(predictions)
    else:
        pred_classes = np.argmax(predictions, axis=1)
    if round_target:
        targets = np.argmax(targets, axis=1)
    conf_matrix = sklearn_confusion_matrix(targets, pred_classes, labels=labels)
    return conf_matrix

def results_from_confusion_matrix(conf_matrix):
    tp = np.diag(conf_matrix)
    fp = conf_matrix.sum(axis=0) - tp
    fn = conf_matrix.sum(axis=1) - tp
    support = conf_matrix.sum(axis=1)

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp, dtype=float), where=(precision + recall) != 0)
    accuracy = tp.sum() / conf_matrix.sum()

    return precision, recall, f1, accuracy

def classification_report_from_confusion_matrix(conf_matrix):
    print("Confusion matrix:")
    print(conf_matrix.astype(int).T)
    
    tp = np.diag(conf_matrix)
    fp = conf_matrix.sum(axis=0) - tp
    fn = conf_matrix.sum(axis=1) - tp
    support = conf_matrix.sum(axis=1)

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) != 0)
    recall = np.divide(tp, tp + fn, out=np.zeros_like(tp, dtype=float), where=(tp + fn) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp, dtype=float), where=(precision + recall) != 0)
    accuracy = tp.sum() / conf_matrix.sum()
    
    report = ""
    report += f"{'Class':<10}{'Precision':<10}{'Recall':<10}{'F1-Score':<10}{'Support':<10}\n"
    report += "-" * 50 + "\n"

    for i in range(len(tp)):
        report += f"{i:<10}{precision[i]:<10.2f}{recall[i]:<10.2f}{f1[i]:<10.2f}{support[i]:<10}\n"

    macro_precision = precision.mean()
    macro_recall = recall.mean()
    macro_f1 = f1.mean()
    weighted_precision = np.average(precision, weights=support)
    weighted_recall = np.average(recall, weights=support)
    weighted_f1 = np.average(f1, weights=support)

    report += "\n"
    report += f"{'Accuracy':<10}{accuracy:<10.2f}{'':<10}{'':<10}{conf_matrix.sum():<10}\n"
    report += f"{'Macro Avg':<10}{macro_precision:<10.2f}{macro_recall:<10.2f}{macro_f1:<10.2f}{support.sum():<10}\n"
    report += f"{'Weigh. Avg':<10}{weighted_precision:<10.2f}{weighted_recall:<10.2f}{weighted_f1:<10.2f}{support.sum():<10}\n"

    return report

def add_conf(conf_aux, conf_global, key):
    if key not in conf_global:
        conf_global[key] = conf_aux.copy()
    else:
        conf_global[key] += conf_aux    

def add_val(reco, true, seq, key):
    if key not in seq:
        seq[key] = {'reco': [reco], 'true': [true]}
    else:
        seq[key]['reco'].append(reco)
        seq[key]['true'].append(true)

In [ ]:
model.eval()

conf_primlepton = {}
conf_seg = {}

t = tqdm.tqdm(enumerate(test_loader), total=len(test_loader), disable=False)


for i, batch in t:

    torch.cuda.empty_cache()
    gc.collect()
    
    # Prepare input and target tensors
    batch_input, batch_input_global, target = _arrange_batch(batch, device)
    
    with torch.no_grad():
        batch_output = model(batch_input, batch_input_global)
    
    # pred
    out_primlepton = [torch.sigmoid(x).detach().cpu().numpy() for x in batch_output['out_primlepton'].decomposed_features]
    out_seg = [torch.softmax(x, dim=1).detach().cpu().numpy() for x in batch_output['out_seg'].decomposed_features]

    batch_size = len(out_primlepton)
    
    # true
    coords = target['coords']
    prim_vertex = target['primary_vertex']
    in_neutrino_pdg = target['in_neutrino_pdg']
    in_neutrino_energy = target['in_neutrino_energy']
    is_cc = target['is_cc']
    targ_primlepton = target['primlepton_labels']
    targ_seg = target['seg_labels']

    for batch_idx in range(batch_size):
        flavour = dataset.pdg2label(in_neutrino_pdg[0], is_cc[0], name=True)
        energy = in_neutrino_energy[batch_idx]
        energy = int(energy//100 * 100)  # transform
        dataset.voxelise(coords[batch_idx], reverse=True)
        distances = np.linalg.norm(coords[batch_idx] - prim_vertex[batch_idx], axis=1)
        
        # primlepton
        conf = confusion_matrix(out_primlepton[batch_idx], targ_primlepton[batch_idx], labels=[0, 1])
        add_conf(conf, conf_primlepton, key="all")
        add_conf(conf, conf_primlepton, key=flavour)
        add_conf(conf, conf_primlepton, key=energy)

        # seg
        conf = confusion_matrix(out_seg[batch_idx], targ_seg[batch_idx], labels=[0, 1, 2], round_target=True)
        add_conf(conf, conf_seg, key="all")
        add_conf(conf, conf_seg, key=flavour)
        add_conf(conf, conf_seg, key=energy)

        # seg close to vertex
        mask = distances<=50
        conf = confusion_matrix(out_seg[batch_idx][mask], targ_seg[batch_idx][mask], labels=[0, 1, 2], round_target=True)
        add_conf(conf, conf_seg, key="<=50mm")
        mask = distances<=100
        conf = confusion_matrix(out_seg[batch_idx][mask], targ_seg[batch_idx][mask], labels=[0, 1, 2], round_target=True)
        add_conf(conf, conf_seg, key="<=100mm")

    del batch_input
    
# (0.9139406313093339, 0.4111643100478804, 0.36962088240746027) same

 16%|█▌        | 428/2717 [1:04:31<4:36:20,  7.24s/it]

In [ ]:
with open('/raid/monsals/faser/results_seg_train.pkl', 'wb') as fd:
    pkl.dump([conf_primlepton, conf_seg], fd)

In [ ]:
with open('/raid/monsals/faser/results_seg_train.pkl', 'rb') as fd:
    conf_primlepton, conf_seg = pkl.load(fd)

In [ ]:
print("#### all ####\n")
print(classification_report_from_confusion_matrix(conf_primlepton['all']))

print("#### nue ####\n")
print(classification_report_from_confusion_matrix(conf_primlepton['CC nue']))

print("#### numu ####\n")
print(classification_report_from_confusion_matrix(conf_primlepton['CC numu']))

print("#### nutau ####\n")
print(classification_report_from_confusion_matrix(conf_primlepton['CC nutau']))

In [ ]:
xticks = np.arange(0, 5800, 100)
yprecision = np.zeros(shape=(xticks.shape[0], 2))
yrecall = np.zeros(shape=(xticks.shape[0], 2))
ydist = np.zeros(shape=xticks.shape[0])

for i, key in enumerate(xticks):
    if key in conf_primlepton:
        precision, recall, f1, accuracy = results_from_confusion_matrix(conf_primlepton[key])
        yprecision[i, :] = precision
        yrecall[i, :] = recall
        ydist[i] = conf_primlepton[key].sum()

yprecision[yprecision==0] = np.nan
yrecall[yrecall==0] = np.nan
yprecision_masked = np.ma.masked_invalid(yprecision)
yrecall_masked = np.ma.masked_invalid(yrecall)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns

# First plot - Precision with Distribution
ax1 = axes[0]
ax1.plot(xticks, yprecision_masked, label=["Rest", "Prim. lepton"])
ax1.set_xlabel("Total neutrino energy (GeV)", fontsize=15)
ax1.set_ylabel("Purity (%)", fontsize=15)
ax1.set_ylim(0, 1.05)
ax1.legend(loc="lower right", fontsize=12)

# Secondary y-axis for the first plot
ax1_secondary = ax1.twinx()
ax1_secondary.fill_between(xticks, ydist, color='orange', alpha=0.3, label='Event distribution (a.u.)')
ax1_secondary.tick_params(axis='y', labelcolor='orange')
ax1_secondary.legend(loc="lower left", fontsize=12)
ax1_secondary.tick_params(axis='y', which='both', left=False, right=False, labelleft=False, labelright=False)

# Second plot - Recall with Distribution
ax2 = axes[1]
ax2.plot(xticks, yrecall_masked, label=["Rest", "Prim. lepton"])
ax2.set_xlabel("Total neutrino energy (GeV)", fontsize=15)
ax2.set_ylabel("Efficiency (%)", fontsize=15)
ax2.set_ylim(0, 1.05)
ax2.legend(loc="lower right", fontsize=12)

# Secondary y-axis for the second plot
ax2_secondary = ax2.twinx()
ax2_secondary.fill_between(xticks, ydist, color='orange', alpha=0.3, label='Event distribution (a.u.)')
ax2_secondary.tick_params(axis='y', labelcolor='orange')
ax2_secondary.legend(loc="lower left", fontsize=12)
ax2_secondary.tick_params(axis='y', which='both', left=False, right=False, labelleft=False, labelright=False)

# Adjust layout
plt.tight_layout()
plt.show()


In [ ]:
print("#### all ####\n")
print(classification_report_from_confusion_matrix(conf_seg['all']))

print("#### <= 100 mm ####\n")
print(classification_report_from_confusion_matrix(conf_seg['<=100mm']))

print("#### <= 50 mm ####\n")
print(classification_report_from_confusion_matrix(conf_seg['<=50mm']))

In [ ]:
xticks = np.arange(0, 5800, 100)
yprecision = np.zeros(shape=(xticks.shape[0], 3))
yrecall = np.zeros(shape=(xticks.shape[0], 3))
ydist = np.zeros(shape=xticks.shape[0])

for i, key in enumerate(xticks):
    if key in conf_seg:
        precision, recall, f1, accuracy = results_from_confusion_matrix(conf_seg[key])
        yprecision[i, :] = precision
        yrecall[i, :] = recall
        ydist[i] = conf_seg[key].sum()

yprecision[yprecision==0] = np.nan
yrecall[yrecall==0] = np.nan
yprecision_masked = np.ma.masked_invalid(yprecision)
yrecall_masked = np.ma.masked_invalid(yrecall)

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns

# First plot - Precision with Distribution
ax1 = axes[0]
ax1.plot(xticks, yprecision_masked, label=["Ghost", "Electromagnetic", "Hadronic"])
ax1.set_xlabel("Total neutrino energy (GeV)", fontsize=15)
ax1.set_ylabel("Purity (%)", fontsize=15)
ax1.set_ylim(0, 1.05)
ax1.legend(loc="lower right", fontsize=12)

# Secondary y-axis for the first plot
ax1_secondary = ax1.twinx()
ax1_secondary.fill_between(xticks, ydist, color='orange', alpha=0.3, label='Event distribution (a.u.)')
ax1_secondary.tick_params(axis='y', labelcolor='orange')
ax1_secondary.legend(loc="lower left", fontsize=12)
ax1_secondary.tick_params(axis='y', which='both', left=False, right=False, labelleft=False, labelright=False)

# Second plot - Recall with Distribution
ax2 = axes[1]
ax2.plot(xticks, yrecall_masked, label=["Ghost", "Electromagnetic", "Hadronic"])
ax2.set_xlabel("Total neutrino energy (GeV)", fontsize=15)
ax2.set_ylabel("Efficiency (%)", fontsize=15)
ax2.set_ylim(0, 1.05)
ax2.legend(loc="lower right", fontsize=12)

# Secondary y-axis for the second plot
ax2_secondary = ax2.twinx()
ax2_secondary.fill_between(xticks, ydist, color='orange', alpha=0.3, label='Event distribution (a.u.)')
ax2_secondary.tick_params(axis='y', labelcolor='orange')
ax2_secondary.legend(loc="lower left", fontsize=12)
ax2_secondary.tick_params(axis='y', which='both', left=False, right=False, labelleft=False, labelright=False)

# Adjust layout
plt.tight_layout()
plt.show()
